# Building mHC from scratch: constraining the widened residual stream
### Manifold-Constrained Hyper-Connections (DeepSeek-AI, arXiv:2512.24880, 31 Dec 2025; v2 5 Jan 2026), one PyTorch step at a time

Residual connections have carried almost every deep network since [He et al., 2016](https://arxiv.org/abs/1512.03385), and almost nobody changed them. [Hyper-Connections](https://arxiv.org/abs/2409.19606) (HC; Zhu et al., 2024) did: they widen the residual stream into `n` parallel streams and let the network learn how to mix them. HC trains to a lower loss than a plain residual, but the mixing matrices are unconstrained, so across many layers their product can amplify or erase the signal, and the extra streams multiply memory traffic. [mHC](https://arxiv.org/abs/2512.24880) (Xie et al., 2025) keeps HC's wider stream and projects the mixing matrices onto the set of **doubly stochastic matrices** with the Sinkhorn-Knopp algorithm, then pays for the extra memory traffic with fused kernels, selective recomputation, and a modified pipeline schedule.

**Headline results from the paper** (27B-parameter mixture-of-experts language models, DeepSeek-V3-style, 262B training tokens, `n = 4`): mHC reaches a final training loss **0.021 lower than the baseline**; on BBH (3-shot exact match) it scores **51.0 vs 43.8 for the baseline and 48.9 for HC**, and on DROP (3-shot F1) **53.9 vs 47.0 and 51.6**; HC shows a loss surge near step 12k that mHC does not; and mHC's measured training-time overhead at `n = 4` is **6.7 %**. We did not rerun any of this.

**What we do here.** We build every piece in PyTorch on a laptop CPU and check each claim we can check at small scale. Everything marked *paper* is quoted from the paper; everything marked *we measured* is produced by the code in this notebook, on models roughly four orders of magnitude smaller than the paper's. The two should not be confused: our toy models show the *mechanism* (unconstrained mixing matrices drift, doubly stochastic ones cannot), not the paper's accuracy or scale.

| section | question | the number it carries |
|---|---|---|
| 1 | What is a widened residual stream, concretely? | a 2-stream example worked by hand |
| 2 | How does an unconstrained mixing matrix go wrong? | composite gain after 60 layers |
| 3 | Why doubly stochastic matrices? | spectral norm, closure, Birkhoff decomposition |
| 4 | How do we make a matrix doubly stochastic? | Sinkhorn-Knopp error after 20 iterations |
| 5 | How are the three mappings parameterized? | parameter overhead per layer |
| 6 | What does a complete layer look like? | equivalence to a plain residual at initialization |
| 7 | Does the stability effect show up when we train? | loss and gradient norm, 3 variants |
| 8 | What does the extra width cost in memory traffic? | elements read/written per token, recompute block size |
| 9 | Closing thoughts | |

*Prerequisites:* transformers and the chain rule. Code is plain PyTorch, CPU only.

In [ ]:
import math, time, itertools, glob, os
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
plt.rcParams.update({"figure.dpi": 105, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25, "font.size": 9})
CHECKS = []
def check(name, ok, detail=""): CHECKS.append((name, bool(ok), detail)); print(("PASS " if ok else "FAIL ") + name + (f"   [{detail}]" if detail else ""))

## Residual streams, widened

A residual layer adds a function of the input back to the input: `x_{l+1} = x_l + F(x_l)` (the paper's Eq. 1). Unrolling this over layers gives `x_L = x_l + Σ_{i=l}^{L-1} F(x_i)` (Eq. 2): the shallow signal `x_l` reaches layer `L` **unchanged**, plus a sum of layer outputs. This *identity mapping* is what He et al. ([2016b](https://arxiv.org/abs/1603.05027)) credit for stable training at depth.

To see what HC changes, take a concrete case with `n = 2` streams and `C = 3` features, so the hidden state is a `2 × 3` matrix:

`x = [[1, 2, 3], [4, 5, 6]]`

HC uses three small matrices per layer (the paper's Eq. 3): a **pre-mapping** `H_pre ∈ R^{1×n}` that reads the streams into one `C`-dimensional layer input, a **post-mapping** `H_post ∈ R^{1×n}` that writes the layer output back to every stream, and a **residual mapping** `H_res ∈ R^{n×n}` that mixes the streams with each other:

`x_{l+1} = H_res · x_l + H_postᵀ · F(H_pre · x_l)`

Let `H_pre = [0.5, 0.5]` (average the streams), `H_post = [1, 1]` (add the layer output to both), `H_res = [[0.9, 0.3], [0.1, 0.8]]`, and `F(u) = 0.1·u`.

In [ ]:
x = torch.tensor([[1., 2., 3.], [4., 5., 6.]])                  # n=2 streams, C=3 features
H_pre = torch.tensor([[0.5, 0.5]]); H_post = torch.tensor([[1., 1.]]); H_res = torch.tensor([[0.9, 0.3], [0.1, 0.8]])
Fn = lambda u: 0.1 * u

h_in = H_pre @ x                                   # (1,n)@(n,C) -> (1,C): what the layer reads
y = Fn(h_in)                                       # (1,C): what the layer computes
x_next = H_res @ x + H_post.T @ y                  # (n,n)@(n,C) + (n,1)@(1,C) -> (n,C)
print("layer input  H_pre @ x       =", h_in.tolist())
print("layer output F(.)            =", y.tolist())
print("residual-stream part H_res@x =", (H_res @ x).tolist())
print("x_next                       =", x_next.tolist())
print("\ncolumn sums of H_res:", H_res.sum(0).tolist(), "  row sums:", H_res.sum(1).tolist())
print("stream-mean of x:", x.mean(0).tolist(), " -> stream-mean of H_res@x:", (H_res @ x).mean(0).tolist())

Read the last two lines. This `H_res` has column sums `1.0, 1.1` and row sums `1.2, 0.9`, so `H_res @ x` does not conserve the average over streams: the mean of `[1, 2, 3]` and `[4, 5, 6]` is `[2.5, 3.5, 4.5]`, and after one application it is `[2.7, 3.75, 4.8]`, between 7 and 8 % larger in each feature. One layer barely matters. The question is what happens when 60 of them are multiplied together. In a plain residual stream the answer is "nothing" (`n = 1`, `H_res = 1`); in HC, `H_res` is a free parameter, and the paper's ablation (Table 1, loss gap relative to the baseline, lower is better) shows that is exactly the part that pays:

| mappings learned | absolute loss gap (*paper*) |
|---|---|
| none (baseline residual) | 0.000 |
| `H_res` only | -0.022 |
| `H_res` + `H_pre` | -0.025 |
| `H_res` + `H_pre` + `H_post` | -0.027 |

So the mixing between streams accounts for 0.022 of the 0.027 loss improvement in the paper's ablation. That is the matrix we will need to keep under control.

<p align="center"><img src="images/fig1_residual_hc_mhc.png" width="760"><br><sub>Fig. 1 (paper). (a) residual; (b) HC adds Res/Pre/Post mappings on an n-stream state; (c) mHC applies a projection P_M to each mapping. Notice that the layer F is unchanged in all three: the entire change lives in the three small matrices around it.</sub></p>

The next section shows how that freedom turns into a stability problem.

## The composite mapping

To see why the free `H_res` is a problem, unroll HC over layers. Dropping the layer outputs for a moment (they are added on top), the residual-stream part of layer `l` multiplies the state by `H_res_l`, so the signal that travels from layer `l` to layer `L` is multiplied by the **composite mapping** `Π_i H_res_{L-i}` (the paper's Eq. 4). In a plain residual stream that product is the identity. In HC it is a product of `L - l` unconstrained `n × n` matrices.

A one-dimensional example shows the scale of the problem. If each layer amplifies by only 10 % (`H_res = 1.1`) the 60-layer product is `1.1^60`; if it attenuates by 10 % it is `0.9^60`.

In [ ]:
print(f"1.1^60 = {1.1 ** 60:,.0f}     0.9^60 = {0.9 ** 60:.5f}")
check("a 10% per-layer gain compounds to >300x over 60 layers; a 10% loss to <0.002x", 1.1 ** 60 > 300 and 0.9 ** 60 < 0.002)

The paper quantifies this with the **Amax gain magnitude** of a (composite) matrix `P`: the **maximum absolute row sum** for the forward pass (how much the largest output stream can grow relative to a unit input in every stream) and the **maximum absolute column sum** for the backward pass (the same for gradients, which flow through `Pᵀ`). A matrix that conserves signal has both equal to 1. In their 27B HC model the composite mapping reaches an Amax gain of roughly **3000** (*paper*, Fig. 3b), and the same run shows a loss surge near step 12k that tracks a rise in gradient norm (Fig. 2).

<p align="center"><img src="images/fig3_hc_gain.png" width="820"><br><sub>Fig. 3 (paper), 27B HC model. Left: single-layer H_res has gains near 1 at most layers. Right: the composite mapping's backward gain climbs by three orders of magnitude through the middle layers. Notice that no single layer looks alarming; the problem is only visible in the product.</sub></p>

We cannot train a 27B model here, but we can simulate the mechanism. We draw 60 per-layer matrices from a family, multiply them in order, and track the Amax gain of the running product, with 4 streams (`n = 4`, the paper's setting). We compare four families:

- **identity**: `H = I` (stable, no mixing between streams);
- **unconstrained**: `H = I + σN` with `N` standard normal (what a free `H_res` initialized at the identity looks like after it has drifted by a random amount `σ` per layer);
- **row-stochastic**: `softmax(κI + σ'N)` along rows (non-negative, rows sum to 1, but columns are free);
- **doubly stochastic**: `Sinkhorn-Knopp(exp(κI + σ'N))` (non-negative, rows *and* columns sum to 1). We derive this operator in Section 4; for now treat it as a function that returns a doubly stochastic matrix.

`σ = 0.15` is a modest per-layer drift (about 15 % per entry). These are *simulated* drifts, not learned ones.

In [ ]:
def sinkhorn_knopp(logits, t_max=20):
    """Alternately normalise columns and rows of exp(logits).  Derived step by step in Section 4."""
    M = logits.exp()
    for _ in range(t_max):
        M = M / M.sum(-2, keepdim=True)      # columns -> sum to 1
        M = M / M.sum(-1, keepdim=True)      # rows    -> sum to 1
    return M

def amax_gain(P):
    """paper's metric: forward = max |row sum|, backward = max |column sum|."""
    return P.abs().sum(-1).amax(-1), P.abs().sum(-2).amax(-1)

def make_layers(kind, L, n, gen, sigma=0.15, kappa=3.0, sigma_c=1.0, t_max=20):
    N = torch.randn(L, n, n, generator=gen); I = torch.eye(n)
    if kind == "identity":        return I.expand(L, n, n).clone()
    if kind == "unconstrained":   return I + sigma * N
    Z = kappa * I + sigma_c * N
    if kind == "row-stochastic":  return torch.softmax(Z, dim=-1)
    if kind == "doubly stochastic": return sinkhorn_knopp(Z, t_max)

def composite_gains(kind, L=60, n=4, seeds=30, **kw):
    fw, bw, mix = [], [], []
    for s in range(seeds):
        H = make_layers(kind, L, n, torch.Generator().manual_seed(s), **kw); P = torch.eye(n); f, b = [], []
        for l in range(L):
            P = H[l] @ P; g = amax_gain(P); f.append(g[0]); b.append(g[1])
        fw.append(torch.stack(f)); bw.append(torch.stack(b))
        A = P.abs(); mix.append((1 - (A.diagonal() / A.sum(1))).mean())
    return torch.stack(fw), torch.stack(bw), torch.stack(mix)

kinds = ["identity", "unconstrained", "row-stochastic", "doubly stochastic"]
G = {k: composite_gains(k) for k in kinds}
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6)); col = dict(zip(kinds, ["#868e96", "#e03131", "#f08c00", "#1c7ed6"]))
for k in kinds:
    axs[0].semilogy(range(1, 61), G[k][0].median(0).values, color=col[k], lw=2, label=k); axs[1].semilogy(range(1, 61), G[k][1].median(0).values, color=col[k], lw=2, label=k)
axs[0].set_title("forward gain: max |row sum| of the running product"); axs[1].set_title("backward gain: max |column sum|")
for a in axs: a.set_xlabel("layers multiplied"); a.set_ylabel("Amax gain (median of 30 seeds)")
axs[0].legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"{'family':20s} {'forward gain @60':>17s} {'backward gain @60':>18s} {'mixing between streams':>24s}")
for k in kinds: print(f"{k:20s} {G[k][0][:, -1].median():17.3g} {G[k][1][:, -1].median():18.3g} {G[k][2].median():24.2f}")
check("unconstrained: composite gain after 60 layers exceeds 10x in the median draw (forward)", G["unconstrained"][0][:, -1].median() > 10, f"{G['unconstrained'][0][:, -1].median():.3g}")
check("doubly stochastic: forward AND backward gain stay within 1.05 after 60 layers", G["doubly stochastic"][0][:, -1].max() < 1.05 and G["doubly stochastic"][1][:, -1].max() < 1.05, f"max fwd {G['doubly stochastic'][0][:, -1].max():.4f}, bwd {G['doubly stochastic'][1][:, -1].max():.4f}")
check("row-stochastic: forward gain is exactly 1 but backward gain grows (> 1.3x)", abs(G["row-stochastic"][0][:, -1].max() - 1) < 1e-4 and G["row-stochastic"][1][:, -1].median() > 1.3, f"bwd {G['row-stochastic'][1][:, -1].median():.2f}")

**What we found.** The unconstrained family behaves as the paper describes in kind: the product's gain grows with depth, in both directions, to a median of roughly 12x after 60 layers at this drift (the draws have a heavy tail, and a larger drift gives far larger gains); the paper's measured ~3000 is from a real 27B run. The identity is perfectly stable, but it never moves information between streams (the "mixing" column is 0), which is the property the paper's Table 1 says is worth `0.022` of loss. **Row-stochastic** matrices (a softmax per row) are the obvious cheap fix and they half-work: the forward gain is exactly 1, because non-negative rows summing to 1 cannot amplify, but nothing bounds the column sums, so the backward gain is free to grow (we measured a median of about 1.4x after 60 layers at this setting, a modest effect that a sharper or longer stack would enlarge). Doubly stochastic matrices bound both at once, and they still mix (the mixing column is non-zero).

These are simulated drifts at one value of `σ`. In a trained network the matrices are not independent random draws, and the paper's measured peak of ~3000 comes from a real 27B run, not from our simulation. What the simulation shows is narrower: **if** the free matrices wander by even ~15 % per layer, 60 layers is enough to produce gains far from 1; the constraint removes that possibility by construction. The next section asks why doubly stochastic matrices have this property.

## Doubly stochastic matrices

A matrix is **doubly stochastic** if its entries are non-negative and every row and every column sums to 1. The paper writes the set of such `n × n` matrices as `M^res` and the constrained mapping as `P_{M^res}(H_res)` (Eq. 6). Here is a concrete one:

```
H = [[0.50, 0.50, 0.00],
     [0.25, 0.25, 0.50],
     [0.25, 0.25, 0.50]]
```

Applied to three streams with values `x = [4, 8, 12]`, each output is a **weighted average** of the inputs: `0.5·4 + 0.5·8 = 6`, `0.25·4 + 0.25·8 + 0.5·12 = 9`, and again `9`. The output `[6, 9, 9]` has the same mean as the input (`8`), because the columns sum to 1, and no output exceeds the largest input, because the rows do too. The paper lists three consequences:

1. **Norm preservation.** The spectral norm is at most 1, so `‖Hx‖₂ ≤ ‖x‖₂`: the map cannot amplify.
2. **Compositional closure.** A product of doubly stochastic matrices is doubly stochastic, so the bound survives any depth.
3. **Birkhoff polytope.** Every doubly stochastic matrix is a convex combination of permutation matrices (Birkhoff-von Neumann theorem), so each layer is a *soft shuffle* of the streams.

When `n = 1` the constraint collapses to the scalar `1`, recovering the plain residual connection. The paper also constrains `H_pre` and `H_post` to be non-negative (via a sigmoid, Section 5) so that positive and negative coefficients cannot cancel each other across layers.

We check all of these directly.

In [ ]:
H = torch.tensor([[.5, .5, 0.], [.25, .25, .5], [.25, .25, .5]]); xs = torch.tensor([4., 8., 12.])
print("H @ x =", (H @ xs).tolist(), "  mean in/out:", xs.mean().item(), (H @ xs).mean().item(), "  rows:", H.sum(1).tolist(), " cols:", H.sum(0).tolist())

torch.manual_seed(1); n = 4; mats = sinkhorn_knopp(torch.randn(2000, n, n) * 1.5, t_max=300)        # 2000 random doubly stochastic matrices (tight convergence)
spec = torch.linalg.matrix_norm(mats, ord=2)
print(f"\nspectral norm over 2000 random doubly stochastic 4x4 matrices: max {spec.max():.6f}  min {spec.min():.4f}")
check("1. norm preservation: spectral norm <= 1 (+1e-5) for 2000 random doubly stochastic matrices", spec.max() <= 1 + 1e-5)
xr = torch.randn(2000, n, 8); gain = (mats @ xr).flatten(1).norm(dim=1) / xr.flatten(1).norm(dim=1)
check("1b. ||Hx||_2 <= ||x||_2 on random inputs", (gain <= 1 + 1e-5).all(), f"max ratio {gain.max():.5f}")
check("1c. stream mean is conserved: mean_over_streams(Hx) == mean_over_streams(x)", torch.allclose((mats @ xr).mean(1), xr.mean(1), atol=1e-5))

P = mats[0]
for k in range(1, 60): P = mats[k] @ P
print(f"product of 60 random doubly stochastic matrices: row sums {P.sum(1).numpy().round(5)}  col sums {P.sum(0).numpy().round(5)}  min entry {P.min():.2e}")
check("2. closure: a 60-layer product is still doubly stochastic (rows, cols = 1 within 1e-4, entries >= 0)", (P.sum(0) - 1).abs().max() < 1e-4 and (P.sum(1) - 1).abs().max() < 1e-4 and P.min() >= 0)
check("2b. n = 1: the constraint reduces to the scalar 1 (plain residual)", torch.allclose(sinkhorn_knopp(torch.randn(1, 1, 1)), torch.ones(1, 1, 1)))

# ---- 3. Birkhoff-von Neumann decomposition by greedy peeling: H = sum_k w_k P_k with P_k permutation matrices ----
def birkhoff(Hm, tol=1e-5):
    Hm = Hm.clone().double(); n = Hm.shape[0]; perms = list(itertools.permutations(range(n))); out = []
    while Hm.max() > tol:
        for p in perms:                                              # find a permutation supported on the positive entries
            if all(Hm[i, p[i]] > tol for i in range(n)): break
        else: break
        w = min(Hm[i, p[i]].item() for i in range(n)); out.append((w, p))
        for i in range(n): Hm[i, p[i]] -= w
    return out
Hm = sinkhorn_knopp(torch.randn(4, 4) * 1.2, t_max=500).double(); dec = birkhoff(Hm)
Rec = sum(w * torch.eye(4, dtype=torch.double)[list(p)] for w, p in dec)
print(f"\nBirkhoff decomposition of a random 4x4 doubly stochastic matrix: {len(dec)} permutations, weights sum to {sum(w for w, _ in dec):.6f}, reconstruction error {(Rec - Hm).abs().max():.1e}")
for w, p in dec[:4]: print(f"  weight {w:.3f}  permutation {p}")
check("3. Birkhoff: H is a convex combination of permutation matrices (weights sum to 1, reconstruction < 1e-3)", abs(sum(w for w, _ in dec) - 1) < 1e-3 and (Rec - Hm).abs().max() < 1e-3, f"{len(dec)} permutations (bound (n-1)^2+1 = 10)")

Each check passes on random matrices, but note what is being tested: they hold for **exactly** doubly stochastic matrices (we used 300 iterations to get close to exact). In practice mHC runs 20 Sinkhorn-Knopp iterations, so the matrices are only *approximately* doubly stochastic, and the next section measures how approximate. First, we need to build the operator.

## Sinkhorn-Knopp, step by step

[Sinkhorn and Knopp (1967)](https://doi.org/10.2140/pjm.1967.21.343) showed that any matrix with strictly positive entries can be scaled to a doubly stochastic one by alternately normalizing its rows and columns, and that the alternation converges. The recipe in the paper (Eq. 9) is:

1. make the matrix positive: `M⁽⁰⁾ = exp(H̃_res)` (an elementwise exponential of unconstrained logits);
2. repeat `t_max` times: `M⁽ᵗ⁾ = T_r(T_c(M⁽ᵗ⁻¹⁾))`, where `T_c` divides every column by its sum and `T_r` divides every row by its sum;
3. the paper uses `t_max = 20`.

Tiny numeric example with `3 × 3` logits `[[0,1,0],[1,0,2],[0,2,0]]`. After each normalization one family of sums is exactly 1 and the other is not; each round shrinks the error in the other.

In [ ]:
Z = torch.tensor([[0., 1., 0.], [1., 0., 2.], [0., 2., 0.]]); M = Z.exp()
print("M0 = exp(Z)\n", M.numpy().round(3), "\n   row sums", M.sum(1).numpy().round(3), " col sums", M.sum(0).numpy().round(3))
for t in range(1, 6):
    M = M / M.sum(0, keepdim=True)          # T_c
    M = M / M.sum(1, keepdim=True)          # T_r
    print(f"t={t}: row sums {M.sum(1).numpy().round(4)}  col sums {M.sum(0).numpy().round(4)}   max|col sum - 1| = {(M.sum(0) - 1).abs().max():.2e}")
print("\nM after 5 iterations:\n", M.numpy().round(4))

The row sums are exactly 1 after every iteration (the row step runs last) and the column sums approach 1: the error falls by roughly a constant factor per round. That raises two practical questions, both of which matter for a method that fixes `t_max = 20`: **how far from doubly stochastic is the output after 20 iterations**, and **how does that error accumulate through 60 layers**? We measure both. We vary the logit scale, because large logits make the matrix close to a permutation (many near-zero entries), which is the slow case for Sinkhorn-Knopp.

In [ ]:
def sk_history(logits, t_max):
    M = logits.exp(); errs = []
    for _ in range(t_max):
        M = M / M.sum(-2, keepdim=True); M = M / M.sum(-1, keepdim=True); errs.append((M.sum(-2) - 1).abs().amax(-1))
    return torch.stack(errs)                 # (t_max, batch): worst column-sum error per matrix

torch.manual_seed(2); scales = [0.5, 1.0, 2.0, 3.0]; hist = {s: sk_history(torch.randn(2000, 4, 4) * s, 50) for s in scales}
fig, ax = plt.subplots(figsize=(6.6, 3.6))
for s in scales: ax.semilogy(range(1, 51), hist[s].median(1).values, label=f"logit std {s}")
ax.axvline(20, color="k", ls=":"); ax.text(21, ax.get_ylim()[1] * 0.3, "t_max = 20 (paper)", fontsize=8); ax.set_xlabel("Sinkhorn-Knopp iterations"); ax.set_ylabel("median worst column-sum error"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"{'logit std':>9s} {'median err @20':>15s} {'99th pct @20':>14s} {'median err @100':>16s}")
h100 = {s: sk_history(torch.randn(2000, 4, 4) * s, 100)[-1] for s in scales}
for s in scales: print(f"{s:9.1f} {hist[s][19].median():15.2e} {hist[s][19].quantile(.99):14.2e} {h100[s].median():16.2e}")
check("with logit std <= 1, 20 iterations give column-sum error below 1e-3 for 99% of matrices", hist[0.5][19].quantile(.99) < 1e-3 and hist[1.0][19].quantile(.99) < 1e-3)
check("sharper matrices converge more slowly: median error at t=20 is larger for logit std 3 than for 0.5", hist[3.0][19].median() > hist[0.5][19].median())

# how the residual error accumulates through depth (paper Fig. 7 shows backward composite gain up to ~1.6 for mHC at 27B)
def depth_gain(t_max, s, L=60, seeds=200):
    out = []
    for sd in range(seeds):
        g = torch.Generator().manual_seed(sd); H = sinkhorn_knopp(torch.randn(L, 4, 4, generator=g) * s, t_max); P = torch.eye(4)
        for l in range(L): P = H[l] @ P
        out.append(amax_gain(P)[1])
    return torch.stack(out)
print(f"\ncomposite backward gain after 60 layers (median / max of 200 draws):")
for s in [1.0, 3.0]:
    a, b = depth_gain(20, s), depth_gain(100, s); print(f"  logit std {s}:  t_max=20 -> {a.median():.3f} / {a.max():.3f}    t_max=100 -> {b.median():.4f} / {b.max():.4f}")

**What we found.** For well-conditioned logits (standard deviation ≤ 1) 20 iterations are plenty: the worst column sum is within `1e-3` of 1 for 99 % of our random `4 × 4` matrices. For sharper matrices the error at `t = 20` is visibly larger, and because the composite mapping multiplies 60 of them, small per-layer errors compound (our depth measurement above). This is consistent with the paper's Fig. 7, where mHC's composite backward gain is bounded but not exactly 1 (it peaks near 1.6 in their 27B run), rather than evidence for a bug. The trade-off is the usual one: more iterations cost more kernel time, fewer leave the gain further from 1. The paper chose 20 as "a practical value" and did not report a sweep, so we cannot say how sensitive their results are to it.

**Numerical safety.** The naive version exponentiates the logits. In `float32`, `exp(x)` overflows to infinity for `x ≳ 88`. A log-domain implementation, which subtracts `logsumexp` instead of dividing, avoids that and is mathematically identical. The paper does not say which form their kernel uses; we show both.

In [ ]:
def sinkhorn_log(logits, t_max=20):
    Zl = logits
    for _ in range(t_max):
        Zl = Zl - torch.logsumexp(Zl, dim=-2, keepdim=True)     # columns
        Zl = Zl - torch.logsumexp(Zl, dim=-1, keepdim=True)     # rows
    return Zl.exp()
Zt = torch.randn(64, 4, 4)
check("log-domain Sinkhorn-Knopp equals the exp-then-divide version", torch.allclose(sinkhorn_log(Zt), sinkhorn_knopp(Zt), atol=1e-6), f"max diff {(sinkhorn_log(Zt) - sinkhorn_knopp(Zt)).abs().max():.1e}")
Zbig = torch.zeros(4, 4); Zbig[0, 0] = 100.0
print("float32 logits with a 100 on the diagonal:  exp-then-divide ->", sinkhorn_knopp(Zbig).isfinite().all().item(), "(finite?)   log-domain ->", sinkhorn_log(Zbig).isfinite().all().item())
check("naive version overflows at logit 100 in float32, the log-domain version does not", (not sinkhorn_knopp(Zbig).isfinite().all()) and sinkhorn_log(Zbig).isfinite().all())

### Differentiating through 20 iterations

The projection sits inside the model, so gradients must flow through it. Every operation is a division or sum, so autograd handles it directly, but it stores a pair of `n × n` tensors per iteration **per token per layer**. The paper's kernel avoids that: "for the backward pass, we derive a custom backward kernel that recomputes the intermediate results on-chip and traverses the entire iteration". We reproduce the idea with a custom `autograd.Function` that saves only the logits and re-runs the iteration in `backward`, and we verify it three ways: against finite differences, against plain autograd, and by counting the bytes autograd saves for backward.

In [ ]:
class SinkhornRecompute(torch.autograd.Function):
    @staticmethod
    def forward(ctx, logits, t_max):
        ctx.save_for_backward(logits); ctx.t = t_max
        with torch.no_grad(): return sinkhorn_knopp(logits, t_max)
    @staticmethod
    def backward(ctx, g):
        (logits,) = ctx.saved_tensors
        with torch.enable_grad():
            z = logits.detach().requires_grad_(True); out = sinkhorn_knopp(z, ctx.t)         # recompute the whole iteration
            (gz,) = torch.autograd.grad(out, z, g)
        return gz, None

torch.manual_seed(3); Zd = torch.randn(4, 4, dtype=torch.double); W = torch.randn(4, 4, dtype=torch.double)
f = lambda z: (sinkhorn_knopp(z, 20) * W).sum()
z1 = Zd.clone().requires_grad_(True); f(z1).backward(); g_auto = z1.grad
z2 = Zd.clone().requires_grad_(True); (SinkhornRecompute.apply(z2, 20) * W).sum().backward(); g_rec = z2.grad
g_fd = torch.zeros_like(Zd); h = 1e-6
for i in range(4):
    for j in range(4):
        e = torch.zeros_like(Zd); e[i, j] = h; g_fd[i, j] = (f(Zd + e) - f(Zd - e)) / (2 * h)
check("gradient through 20 Sinkhorn-Knopp iterations matches finite differences", torch.allclose(g_auto, g_fd, atol=1e-7), f"max err {(g_auto - g_fd).abs().max():.1e}")
check("recompute-in-backward gradient equals plain autograd", torch.allclose(g_rec, g_auto, atol=1e-12), f"max err {(g_rec - g_auto).abs().max():.1e}")

def saved_bytes(fn):
    tot = [0]
    def pack(t): tot[0] += t.numel() * t.element_size(); return t
    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        out = fn()                                   # count what the FORWARD keeps alive for backward (backward's own recompute is transient)
    out.sum().backward()
    return tot[0]
Zf = torch.randn(1024, 4, 4, requires_grad=True)
b_plain = saved_bytes(lambda: sinkhorn_knopp(Zf, 20)); b_rec = saved_bytes(lambda: SinkhornRecompute.apply(Zf, 20))
print(f"bytes saved for backward, 1024 matrices, 20 iterations:  plain autograd {b_plain:,}   recompute {b_rec:,}   ({b_plain / b_rec:.0f}x less)")
check("recomputing in backward saves >10x activation memory for the Sinkhorn-Knopp step", b_plain / b_rec > 10, f"{b_plain / b_rec:.0f}x")

The cost is one extra forward pass through a tiny `n × n` iteration during backward, which is cheap next to the layer itself; the paper implements this on-chip in a single kernel. We have the operator and its gradient. Next we need to decide what to feed it.

## Three mappings, three parameterizations

mHC computes all three mappings from the **current hidden state**, so they are *dynamic* (per token) plus a *static* learned bias (the paper's Eq. 7-8). For a hidden state `x_l ∈ R^{n×C}`:

1. flatten to `x⃗ ∈ R^{1×nC}` and apply RMSNorm (so the coefficients see the whole stream, not one stream);
2. project: `H̃_pre = α_pre · (x⃗'φ_pre) + b_pre`, `H̃_post = α_post · (x⃗'φ_post) + b_post`, `H̃_res = α_res · mat(x⃗'φ_res) + b_res`, with `φ_pre, φ_post ∈ R^{nC×n}`, `φ_res ∈ R^{nC×n²}`, and learnable scalar gates `α` initialized to **0.01** (paper, Table 5);
3. constrain: `H_pre = σ(H̃_pre)`, `H_post = 2σ(H̃_post)`, `H_res = Sinkhorn-Knopp(H̃_res)`.

The sigmoid keeps `H_pre ∈ (0, 1)` and `H_post ∈ (0, 2)`, both non-negative, so contributions cannot cancel. The factor 2 lets `H_post` reach 1 at logit 0, which is the "plain residual" value.

**Initialization.** The paper's text gives the gate `α = 0.01` but not the bias or `φ` initialization. We chose the following, which makes a fresh layer behave like a plain residual (an assumption we state rather than a replication of their code):

| mapping | what we want at init | our logit init | resulting value |
|---|---|---|---|
| `H_pre` | average the `n` streams (`1/n`) | `b_pre = -log(n-1)` | `σ(b) = 1/n` |
| `H_post` | add the layer output to every stream (`1`) | `b_post = 0` | `2σ(0) = 1` |
| `H_res` | identity | `b_res = κ·I`, `κ = 10` | Sinkhorn-Knopp(exp(10·I)) ≈ I |
| `φ_*` | small | `N(0, 0.02²)` | dynamic part ≈ 0 given `α = 0.01` |

These match the fixed mappings the paper uses in its Table 1 ablation (uniform `1/n` for `H_pre`, ones for `H_post`, identity for `H_res`).

In [ ]:
def rms(x, eps=1e-6): return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps)      # RMSNorm without a weight (paper absorbs the weight into phi)

class Coeffs(nn.Module):
    """Eq. (7)-(8): H_pre (1xn), H_post (1xn), H_res (nxn) for every token.  mode='mhc' constrains them, mode='hc' leaves them free."""
    def __init__(s, n, C, mode="mhc", kappa=10.0, t_max=20, alpha0=0.01):
        super().__init__(); s.n, s.mode, s.t_max = n, mode, t_max
        s.phi_pre = nn.Parameter(torch.randn(n * C, n) * .02); s.phi_post = nn.Parameter(torch.randn(n * C, n) * .02); s.phi_res = nn.Parameter(torch.randn(n * C, n * n) * .02)
        s.a_pre, s.a_post, s.a_res = (nn.Parameter(torch.tensor(alpha0)) for _ in range(3))
        if mode == "mhc":      # biases are LOGITS
            s.b_pre = nn.Parameter(torch.full((n,), -math.log(n - 1) if n > 1 else 0.)); s.b_post = nn.Parameter(torch.zeros(n)); s.b_res = nn.Parameter(kappa * torch.eye(n))
        else:                  # unconstrained HC: the biases are the coefficients themselves
            s.b_pre = nn.Parameter(torch.full((n,), 1. / n)); s.b_post = nn.Parameter(torch.ones(n)); s.b_res = nn.Parameter(torch.eye(n))
    def forward(s, x):                                                       # x (B,T,n,C)
        B, T, n, C = x.shape; xv = rms(x.reshape(B, T, n * C))              # flatten the n streams, normalise
        pre = s.a_pre * (xv @ s.phi_pre) + s.b_pre; post = s.a_post * (xv @ s.phi_post) + s.b_post
        res = s.a_res * (xv @ s.phi_res).view(B, T, n, n) + s.b_res          # mat(.): reshape n^2 -> n x n
        if s.mode == "mhc": return torch.sigmoid(pre), 2 * torch.sigmoid(post), sinkhorn_knopp(res, s.t_max)
        return pre, post, res

n, C = 4, 32; co = Coeffs(n, C); xin = torch.randn(2, 5, n, C)
Hp, Ho, Hr = co(xin)
print("shapes  H_pre", tuple(Hp.shape), " H_post", tuple(Ho.shape), " H_res", tuple(Hr.shape))
print("at init  H_pre ~", Hp[0, 0].detach().numpy().round(3), " H_post ~", Ho[0, 0].detach().numpy().round(3), "\n         H_res[0,0] =\n", Hr[0, 0].detach().numpy().round(4))
check("H_pre in (0,1), H_post in (0,2), H_res non-negative", bool((Hp > 0).all() and (Hp < 1).all() and (Ho > 0).all() and (Ho < 2).all() and (Hr >= 0).all()))
check("H_res rows sum to 1 exactly and columns to 1 within 1e-3 after 20 iterations", (Hr.sum(-1) - 1).abs().max() < 1e-5 and (Hr.sum(-2) - 1).abs().max() < 1e-3, f"col err {(Hr.sum(-2) - 1).abs().max():.1e}")
check("at initialization H_pre = 1/n, H_post = 1, H_res = I (up to 5e-3; the small dynamic part moves them by ~1e-3)", torch.allclose(Hp, torch.full_like(Hp, 1 / n), atol=5e-3) and torch.allclose(Ho, torch.ones_like(Ho), atol=5e-3) and torch.allclose(Hr, torch.eye(n).expand_as(Hr), atol=5e-3))

# ---- cost of the coefficients: parameters per layer = nC(2n + n^2) + n^2 + 2n + 3 ----
def coeff_params(n, C): return n * C * (2 * n + n * n) + n * n + 2 * n + 3
check("parameter formula nC(2n+n^2)+n^2+2n+3 matches the module", sum(p.numel() for p in co.parameters()) == coeff_params(n, C))
n27, C27, L27 = 4, 2560, 30                                      # 27B model in the paper's Table 5: width 2560, 30 layers (attention + FFN each carry their own mappings -> 60)
extra = 2 * L27 * coeff_params(n27, C27); print(f"\n27B config (paper Table 5): {extra / 1e6:.1f}M extra parameters = {100 * extra / 27.0e9:.3f}% of 27.0B total, {100 * extra / 4.14e9:.2f}% of 4.14B active")
fl = 2 * 2 * L27 * n27 * C27 * (n27 * n27 + 2 * n27); print(f"coefficient matmul FLOPs per token (fwd): {fl / 1e6:.1f}M vs ~{2 * 4.14e9 / 1e9:.1f}G for the active model = {100 * fl / (2 * 4.14e9):.2f}%")

So the coefficients add about **0.05 % of the parameters** and, by our estimate, **under 0.5 % of the forward FLOPs** at the paper's 27B configuration (we computed these from the paper's Table 5 dimensions; the paper itself only says the cost is "negligible"). The paper's *measured* overhead is **6.7 % wall-clock**, an order of magnitude larger than the FLOP share. That gap is the memory-traffic problem of Section 8: the arithmetic is small, but the wider stream has to be read and written. First, we assemble the layer.

## A complete layer

With coefficients in hand, the layer is three lines (Eq. 3 with the constrained mappings):

```python
h_in = H_pre  @ x          # (1,n)@(n,C) -> (1,C)   read the streams
y    = F(h_in)             # the usual attention or FFN block, unchanged
x    = H_res  @ x  +  H_post.T @ y     # mix the streams, write the output back to all of them
```

We batch it with `einsum` over tokens, wrap any `F` in it, and assemble a small pre-norm language model whose attention and feed-forward blocks are each wrapped (the paper's Fig. 3 counts them as separate layers, which is why a 30-layer model has 60 mixing layers). Two design choices we made and the paper's text does not specify: the model starts with the embedding copied into all `n` streams, and ends by **summing** the streams before the output head. We also include the unconstrained `hc` mode, parameterized identically (same `φ`, `α`, RMSNorm) but without the sigmoid and Sinkhorn-Knopp projections, so the only difference between `hc` and `mhc` in our experiments is the constraint.

In [ ]:
class StreamLayer(nn.Module):
    def __init__(s, Fn, n, C, mode, **kw):
        super().__init__(); s.F, s.mode, s.n = Fn, mode, n
        if mode != "res": s.co = Coeffs(n, C, mode, **kw)
    def forward(s, x, return_res=False):
        if s.mode == "res": return x + s.F(x)                                          # x (B,T,C): the ordinary residual
        pre, post, res = s.co(x)                                                        # x (B,T,n,C)
        h = torch.einsum("btn,btnc->btc", pre, x); y = s.F(h)
        out = torch.einsum("btnm,btmc->btnc", res, x) + torch.einsum("btn,btc->btnc", post, y)
        return (out, res) if return_res else out
class Attn(nn.Module):
    def __init__(s, C, h): super().__init__(); s.h = h; s.qkv = nn.Linear(C, 3 * C, bias=False); s.o = nn.Linear(C, C, bias=False)
    def forward(s, x):
        B, T, C = x.shape; q, k, v = s.qkv(rms(x)).view(B, T, 3, s.h, C // s.h).permute(2, 0, 3, 1, 4)
        return s.o(F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, C))
class FFN(nn.Module):
    def __init__(s, C): super().__init__(); s.a = nn.Linear(C, 4 * C, bias=False); s.b = nn.Linear(4 * C, C, bias=False)
    def forward(s, x): return s.b(F.gelu(s.a(rms(x))))
class LM(nn.Module):
    def __init__(s, V, T, C=64, blocks=8, h=4, n=4, mode="res", **kw):
        super().__init__(); s.mode, s.n = mode, n; s.emb = nn.Embedding(V, C); s.pos = nn.Embedding(T, C); s.layers = nn.ModuleList()
        for _ in range(blocks): s.layers.append(StreamLayer(Attn(C, h), n, C, mode, **kw)); s.layers.append(StreamLayer(FFN(C), n, C, mode, **kw))
        s.head = nn.Linear(C, V, bias=False)
    def forward(s, idx, return_res=False):
        B, T = idx.shape; x = s.emb(idx) + s.pos(torch.arange(T))
        if s.mode != "res": x = x[:, :, None, :].expand(-1, -1, s.n, -1).contiguous()    # copy the embedding into n streams
        ress = []
        for l in s.layers:
            if s.mode != "res" and return_res: x, r = l(x, True); ress.append(r)
            else: x = l(x)
        if s.mode != "res": x = x.sum(2)                                                   # collapse the n streams by summation
        lo = s.head(rms(x)); return (lo, ress) if return_res else lo

# ---- checks: (a) fresh mHC/HC layers reproduce the plain residual model, (b) gradients reach every coefficient ----
V_, T_ = 96, 32; torch.manual_seed(0)
m_res, m_mhc, m_hc = LM(V_, T_, mode="res"), LM(V_, T_, mode="mhc"), LM(V_, T_, mode="hc")
for m in (m_mhc, m_hc):                                                                   # share the weights of F, embeddings and head
    m.emb.load_state_dict(m_res.emb.state_dict()); m.pos.load_state_dict(m_res.pos.state_dict()); m.head.load_state_dict(m_res.head.state_dict())
    for a_, b_ in zip(m_res.layers, m.layers): b_.F.load_state_dict(a_.F.state_dict())
idx = torch.randint(0, V_, (4, T_))
with torch.no_grad(): lr_, lm_, lh_ = m_res(idx), m_mhc(idx), m_hc(idx)
print(f"mean |logit| = {lr_.abs().mean():.3f};  max |res - mHC| = {(lr_ - lm_).abs().max():.2e};  max |res - HC| = {(lr_ - lh_).abs().max():.2e}")
check("at initialization an mHC model computes (almost) the same function as the plain residual model (max logit diff < 1e-2)", (lr_ - lm_).abs().max() < 1e-2, f"{(lr_ - lm_).abs().max():.1e}")
lo, ress = m_mhc(idx, return_res=True); lo.sum().backward()
dead = [n_ for n_, p in m_mhc.named_parameters() if p.grad is None or p.grad.abs().sum() == 0]
check("every mHC parameter (phi, alpha, bias) receives a non-zero gradient", len(dead) == 0, f"{len(dead)} without: {dead[:3]}")
check("one H_res per mixing layer: 16 for 8 blocks (attention + FFN each)", len(ress) == 16)
check("n = 1 mHC layer has H_res == 1 exactly (the plain residual)", torch.allclose(Coeffs(1, 8)(torch.randn(1, 3, 1, 8))[2], torch.ones(1, 3, 1, 1)))
print(f"parameters: residual {sum(p.numel() for p in m_res.parameters()):,}   mHC {sum(p.numel() for p in m_mhc.parameters()):,}  (+{100 * (sum(p.numel() for p in m_mhc.parameters()) / sum(p.numel() for p in m_res.parameters()) - 1):.0f}% at this tiny width C=64; the share shrinks as C grows)")

At initialization the mHC model reproduces the plain residual model to within `1e-2` in the logits, so any difference in training comes from what the coefficients learn, not from a different starting function. (The HC model also starts close to it, at a larger distance; its coefficients are free from the first step.) The model is ready for the question that motivated all of this: does it train differently?

## Stability at toy scale

We now train three variants of the same small language model: the plain **residual** baseline (`res`), unconstrained **Hyper-Connections** (`hc`), and **mHC** (`mhc`). All three share the architecture of Section 6 (8 blocks = 16 mixing layers, width `C = 64`, `n = 4` streams for `hc` and `mhc`), the same data order and the same optimizer (AdamW, betas `(0.9, 0.95)`, weight decay 0.1, 50-step linear warmup, gradient norm recorded before any clipping, and no clipping). The corpus is character-level Python source from the installed `torch` package (`torch/nn/modules` and `torch/optim`), a convenient offline text; the held-out set is the last 10 % of the character stream.

This setting differs from the paper's in every way that matters for scale: roughly 10⁻⁴ of the parameters, a few hundred optimizer steps instead of 50 000, dense rather than mixture-of-experts layers, and characters rather than tokens. The paper's instability appears after ~12k steps at 27B. We should not expect to see a loss surge here, and we do not claim to. What we can measure is (a) whether the three variants train at comparable speed at two learning rates, (b) their gradient norms, and (c) the Amax gain of the **learned** composite `H_res` product, which is the quantity the paper uses to diagnose HC's instability.

In [ ]:
import torch as _t
_root = os.path.dirname(_t.__file__)
_files = sorted(glob.glob(_root + "/nn/modules/*.py")) + sorted(glob.glob(_root + "/optim/*.py"))
text = "".join(open(f, errors="ignore").read() for f in _files)
chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}; data = torch.tensor([stoi[c] for c in text])
V, T, ntr = len(chars), 128, int(0.9 * len(data)); train_d, val_d = data[:ntr], data[ntr:]
print(f"corpus: {len(data):,} characters, vocabulary {V}; train {len(train_d):,}  held-out {len(val_d):,}")

def get_batch(d, B, g):
    i = torch.randint(0, len(d) - T - 1, (B,), generator=g); return torch.stack([d[j:j + T] for j in i]), torch.stack([d[j + 1:j + T + 1] for j in i])

def run(mode, lr, steps, blocks=8, B=16, seed=0):
    torch.manual_seed(seed); g = torch.Generator().manual_seed(seed); m = LM(V, T, C=64, blocks=blocks, mode=mode)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1); losses, gnorms = [], []; t0 = time.time()
    for s in range(steps):
        for pg in opt.param_groups: pg["lr"] = lr * min(1.0, (s + 1) / 50)
        x, y = get_batch(train_d, B, g); loss = F.cross_entropy(m(x).reshape(-1, V), y.reshape(-1)); opt.zero_grad(); loss.backward()
        gnorms.append(torch.nn.utils.clip_grad_norm_(m.parameters(), 1e9).item()); opt.step(); losses.append(loss.item())
        if not math.isfinite(losses[-1]): break
    xv, yv = get_batch(val_d, 64, torch.Generator().manual_seed(123))
    with torch.no_grad(): vl = F.cross_entropy(m(xv).reshape(-1, V), yv.reshape(-1)).item()
    return dict(model=m, losses=np.array(losses), gnorms=np.array(gnorm_clean(gnorms)), val=vl, time=time.time() - t0, steps=len(losses))
def gnorm_clean(g): return [x if math.isfinite(x) else float("nan") for x in g]

In [ ]:
STEPS, LRS, MODES = 250, [2e-3, 8e-3], ["res", "hc", "mhc"]
R = {}
for lr in LRS:
    for mode in MODES:
        R[(mode, lr)] = run(mode, lr, STEPS); r = R[(mode, lr)]
        print(f"lr {lr:g}  {mode:3s}  train loss (last 50) {r['losses'][-50:].mean():.3f}   held-out {r['val']:.3f}   max grad-norm {np.nanmax(r['gnorms']):7.2f}   {r['time']:.0f}s", flush=True)
fig, axs = plt.subplots(2, 2, figsize=(11, 6)); cmap = {"res": "#868e96", "hc": "#e03131", "mhc": "#1c7ed6"}; sm = lambda x, k=10: np.convolve(x, np.ones(k) / k, mode="valid")
for j, lr in enumerate(LRS):
    for mode in MODES: axs[0, j].plot(sm(R[(mode, lr)]["losses"]), color=cmap[mode], label=mode); axs[1, j].semilogy(R[(mode, lr)]["gnorms"], color=cmap[mode], alpha=.8, label=mode)
    axs[0, j].set_title(f"training loss, lr {lr:g}"); axs[1, j].set_title(f"gradient norm, lr {lr:g}"); axs[1, j].set_xlabel("step"); axs[0, j].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ---- the paper's diagnostic on the LEARNED matrices: composite H_res product, Amax gain, averaged over tokens ----
@torch.no_grad()
def learned_composite_gain(m, n_tok_batch=8):
    g = torch.Generator().manual_seed(7); x, _ = get_batch(val_d, n_tok_batch, g); _, ress = m(x, return_res=True)     # list of (B,T,n,n), one per mixing layer
    P = None; fw, bw = [], []
    for H in ress:
        P = H if P is None else H @ P; a, b = amax_gain(P); fw.append(a.mean().item()); bw.append(b.mean().item())
    return np.array(fw), np.array(bw)
print(f"{'variant':10s} {'lr':>7s} {'composite forward gain':>24s} {'composite backward gain':>25s}   (after all 16 mixing layers; mean over tokens)")
LG = {}
for lr in LRS:
    for mode in ["hc", "mhc"]:
        fw, bw = learned_composite_gain(R[(mode, lr)]["model"]); LG[(mode, lr)] = (fw, bw); print(f"{mode:10s} {lr:7g} {fw[-1]:24.4f} {bw[-1]:25.4f}")
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for (mode, lr), (fw, bw) in LG.items(): ax.plot(range(1, 17), np.maximum(fw, bw), color=cmap[mode], ls="-" if lr == LRS[0] else "--", label=f"{mode}, lr {lr:g}")
ax.set_xlabel("mixing layers multiplied"); ax.set_ylabel("max(forward, backward) Amax gain"); ax.legend(fontsize=7); ax.set_title("learned composite H_res (cf. paper Fig. 3b / 7b)"); plt.tight_layout(); plt.show()
check("mHC's learned composite gain stays within 1.1 of 1 (forward and backward) at both learning rates", all(max(LG[("mhc", lr)][0][-1], LG[("mhc", lr)][1][-1]) < 1.1 for lr in LRS), f"{[round(max(LG[('mhc', lr)][0][-1], LG[('mhc', lr)][1][-1]), 4) for lr in LRS]}")
check("every variant trains: final train loss below the first-step loss at both learning rates", all(R[k]["losses"][-50:].mean() < R[k]["losses"][:5].mean() for k in R if np.isfinite(R[k]["losses"]).all()))

**What we found.**

- *Training speed and loss.* At both learning rates the three variants reach nearly the same loss: held-out loss `2.207 / 2.184 / 2.205` (residual / HC / mHC) at `lr = 2e-3` and `2.132 / 2.110 / 2.116` at `lr = 8e-3`. Differences of 0.02 are inside what a different seed would change (one seed, 250 steps), so we make no ranking claim.
- *Gradient norm.* At `lr = 8e-3` the unconstrained HC model's largest gradient norm is `4.06`, more than twice the residual model's and mHC's (both `1.78`); at `lr = 2e-3` all three are within 0.1 of each other.
- *The paper's diagnostic on the learned matrices.* At `lr = 8e-3` the composite `H_res` product learned by the unconstrained HC model has a forward gain of about `44.5` and a backward gain of about `53.0` after 16 mixing layers, against `1.00` and `1.07` for mHC (at `lr = 2e-3` HC is at `1.24` / `1.19`). So in a 16-layer, 64-wide model trained for 250 steps, **the unconstrained matrices already drift far enough at the higher learning rate to produce gains around 50**, and the doubly stochastic ones do not. That is the mechanism the paper describes (its Fig. 3 reports ~3000 at 27B after tens of thousands of steps), seen at toy scale; it did not show up as a loss surge or a loss difference in these runs, which is also what the paper's account would predict for short runs at small scale (their surge appears around step 12k).

Three cautions. One seed and 250 steps: the gain values depend on the run. The mHC backward gain of `1.07` at the higher learning rate is the approximate-Sinkhorn residual of Section 4 (20 iterations), not a violation of the constraint. And the constraint holding by construction at any scale means the mHC half of the comparison is expected; the informative half is how far the unconstrained model drifted.

Next, we look at what the paper reports at 27B, and then at the cost side of the ledger.

## The memory wall

On the FLOP count mHC is nearly free (Section 5). On a GPU it is not, because a residual add is **memory-bound**: the time goes into reading and writing activations, not into arithmetic ([Dao et al., 2022](https://arxiv.org/abs/2205.14135) call this the memory wall). Widening the stream to `n` copies multiplies exactly the traffic that was already the bottleneck.

Tiny example: a plain residual layer reads `x` and `F(x)` (`2C` elements) and writes the sum (`C`), so `3C` elements per token. With `C = 2560` that is 7 680 elements. The paper's Table 2 counts the per-token traffic of HC's coefficient computation and application, **excluding** the layer `F` itself, and finds that it grows roughly in proportion to `n`. We reproduce the table from its row definitions and check the totals against the paper's closed forms `(5n+1)C + n² + 2n` (read) and `(3n+1)C + n² + 2n` (write).

In [ ]:
def hc_io_rows(n, C):                    # (reads, writes) per token for each HC operation, paper Table 2
    return {"calculate H_pre,H_post,H_res": (n * C, n * n + 2 * n), "apply H_pre": (n * C + n, C), "apply H_post": (C + n, n * C),
            "apply H_res": (n * C + n * n, n * C), "residual merge": (2 * n * C, n * C)}
n, C = 4, 2560
rows = hc_io_rows(n, C); rd = sum(r for r, _ in rows.values()); wr = sum(w for _, w in rows.values())
check("row sums reproduce the paper's closed forms (5n+1)C+n^2+2n and (3n+1)C+n^2+2n", rd == (5 * n + 1) * C + n * n + 2 * n and wr == (3 * n + 1) * C + n * n + 2 * n)
res_rd, res_wr = 2 * C, C
print(f"n={n}, C={C}  (elements per token per layer, excluding F)")
print(f"  plain residual : read {res_rd:>7,}  write {res_wr:>7,}  total {res_rd + res_wr:>7,}")
print(f"  HC / unfused   : read {rd:>7,}  write {wr:>7,}  total {rd + wr:>7,}   = {(rd + wr) / (res_rd + res_wr):.1f}x the residual")
# the paper fuses the application of H_post, H_res and the merge into one kernel: reads (3n+1)C -> (n+1)C, writes 3nC -> nC
fused_rd = rd - ((3 * n + 1) * C) + (n + 1) * C; fused_wr = wr - 3 * n * C + n * C
print(f"  after fusing only that one kernel (our arithmetic from the paper's description): read {fused_rd:,}  write {fused_wr:,}  total {fused_rd + fused_wr:,} = {(fused_rd + fused_wr) / (res_rd + res_wr):.1f}x")
check("fusing post/res/merge cuts HC traffic by ~half (our arithmetic)", 0.4 < (fused_rd + fused_wr) / (rd + wr) < 0.6, f"{(fused_rd + fused_wr) / (rd + wr):.2f}")
print("\nn   HC total / residual total")
for n_ in [1, 2, 4, 8]: r_ = hc_io_rows(n_, C); tot = sum(a + b for a, b in r_.values()); print(f"{n_}   {tot / (3 * C):.1f}x")
check("HC traffic grows roughly linearly in n (ratio at n=8 is within 1.6-2.4x of the ratio at n=4)", 1.6 < sum(a + b for a, b in hc_io_rows(8, C).values()) / sum(a + b for a, b in hc_io_rows(4, C).values()) < 2.4)

Fusion is the first of three system measures in the paper (Section 4.3). (1) **Kernel fusion**: the normalization divide is moved after the matrix multiply (mathematically equivalent), the `n²+2n` coefficients come out of one fused matmul in mixed precision (`φ` in tfloat32, activations in bfloat16, coefficients and Sinkhorn-Knopp in float32), and the application of `H_post`, `H_res` and the residual merge is a single kernel; most kernels are written in TileLang (Wang et al., 2025). (2) **Recomputation**: the `n`-stream activations are large, so the intermediate results of the mHC kernels are thrown away after the forward pass and recomputed in the backward pass, *without* re-running the heavy layer `F`. (3) **Pipeline communication**: with pipeline parallelism, each stage boundary now sends `n` times more activations; the paper extends the DualPipe schedule so that communication overlaps with compute, and runs the MLP-side `F_post,res` kernels on a high-priority stream.

<p align="center"><img src="images/fig4_dualpipe.png" width="900"><br><sub>Fig. 4 (paper). The extended DualPipe schedule. Notice the third row: a dedicated high-priority compute stream for the F_post,res kernels, so they are not queued behind long attention kernels while communication is waiting.</sub></p>

Recomputation introduces a design parameter we can work out ourselves. If we recompute in blocks of `L_r` consecutive layers, we store one `nC` input per block (`⌈L/L_r⌉` of them) and hold a transient `(n+2)C × L_r` for the active block during backward, so the footprint is `nC·⌈L/L_r⌉ + (n+2)C·L_r`, minimized near `L_r* = √(nL / (n+2))` (the paper's Eq. 20). Two worked cases: `n = 4`, with `L = 60` mixing layers (the 27B model counted as in Fig. 3) and with `L = 30`.

In [ ]:
def recompute_mem(Lr, L, n): return n * math.ceil(L / Lr) + (n + 2) * Lr             # in units of C elements per token
print(f"{'L':>3s} {'formula L_r*':>13s} {'best integer L_r':>17s} {'mem at best':>12s} {'mem at L_r=1 (store every layer)':>34s}")
for L in [30, 60]:
    n = 4; best = min(range(1, L + 1), key=lambda r: recompute_mem(r, L, n)); fr = math.sqrt(n * L / (n + 2))
    print(f"{L:3d} {fr:13.2f} {best:17d} {recompute_mem(best, L, n):12d} {recompute_mem(1, L, n):34d}   ({recompute_mem(1, L, n) / recompute_mem(best, L, n):.1f}x smaller)")
    check(f"L={L}: the best integer block size is within 1 of sqrt(nL/(n+2))", abs(best - fr) <= 1, f"{best} vs {fr:.2f}")
lens = np.arange(1, 31); plt.figure(figsize=(6, 3.2)); plt.plot(lens, [recompute_mem(r, 60, 4) for r in lens], "o-", ms=3, label="L = 60"); plt.plot(lens, [recompute_mem(r, 30, 4) for r in lens], "s-", ms=3, label="L = 30")
plt.xlabel("recompute block size L_r"); plt.ylabel("elements per token / C"); plt.legend(); plt.title("stored + transient memory vs block size (n = 4)"); plt.tight_layout(); plt.show()

The measured outcome in the paper is a **6.7 % training-time overhead** at `n = 4` for in-house large-scale training, against a baseline we are not told the exact configuration of. The three measures are what turn a ~11× traffic increase (our count above, unfused) into that figure; we cannot separate how much each contributes because the paper reports no per-measure ablation. We also did not write any GPU kernels: everything in this notebook runs eagerly in PyTorch on CPU, so we make **no** claim about our own implementation's speed.

Having seen what the method costs, we return to what it buys.

## What the paper reports, and what we can say about it

All numbers in this section are from the paper (27B mixture-of-experts models trained with `n = 4`; baseline = same architecture with ordinary residuals). We did not rerun any of them.

<p align="center"><img src="images/fig5_stability.png" width="820"><br><sub>Fig. 5 (paper), 27B. Left: loss gap relative to the baseline (more negative is better). Right: gradient norm. Notice the middle curve: HC (light blue) ends closer to the baseline than mHC, and its gradient norm is the noisiest of the three, while mHC's tracks the baseline.</sub></p>

| benchmark (shots) | baseline | HC | mHC |
|---|---|---|---|
| BBH, EM (3) | 43.8 | 48.9 | **51.0** |
| DROP, F1 (3) | 47.0 | 51.6 | **53.9** |
| GSM8K, EM (8) | 46.7 | 53.2 | **53.8** |
| HellaSwag, acc (10) | 73.7 | 74.3 | **74.7** |
| MATH, EM (4) | 22.0 | **26.4** | 26.0 |
| MMLU, acc (5) | 59.0 | 63.0 | **63.4** |
| PIQA, acc (0) | 78.5 | 79.9 | **80.5** |
| TriviaQA, EM (5) | 54.3 | 56.3 | **57.6** |

mHC beats the baseline on all eight and HC on seven of eight (MATH is the exception, 26.0 vs 26.4). The paper reports no seeds or confidence intervals, so differences of a few tenths of a point, such as MATH, should not be over-read. For scale: the compute scaling curve (3B, 9B, 27B) shows the advantage "robustly maintained ... showing only marginal attenuation", and the token-scaling run (3B, 1T tokens) shows the same trend.

<p align="center"><img src="images/fig6_scaling.png" width="820"><br><sub>Fig. 6 (paper). Left: compute scaling; right: token scaling. Notice that the gap to the baseline narrows slightly with compute (the paper's 'marginal attenuation'), so we should not extrapolate the 27B gain to larger models.</sub></p>

<p align="center"><img src="images/fig7_mhc_gain.png" width="820"><br><sub>Fig. 7 (paper). The same measurement as Fig. 3 for mHC. Notice the y-axis: 0 to 2 instead of 10^5. The backward composite gain still peaks near 1.6, which is what approximate (20-iteration) Sinkhorn-Knopp predicts, as we measured in Section 4.</sub></p>

**What our own measurements add.** Sections 2-4 reproduce the *mechanism* behind Fig. 3 and Fig. 7 on simulated drifts; Section 7 shows it on learned matrices in a small trained model (gains around 50 for the unconstrained HC model at the higher learning rate, 1.0-1.07 for mHC). The comparison is on our toy scale, in a few hundred steps, with one seed, and cannot support or contradict the paper's 27B results.

## Closing thoughts

Residual connections have been the least-changed part of the transformer: [Highway networks](https://arxiv.org/abs/1505.00387) (Srivastava et al., 2015) added gates to the skip path, ResNet removed them ([He et al., 2016](https://arxiv.org/abs/1512.03385)), and DenseNet-style dense connectivity widened them for vision networks. HC (Zhu et al., 2024) reopened the question with a learnable, wider residual stream; mHC (Xie et al., 2025) answers a specific objection to it, namely that learnable mixing of streams breaks the identity-mapping guarantee, by restricting the mixing to the Birkhoff polytope, where the guarantee comes back by construction, and by treating the extra memory traffic as a first-class engineering problem.

**What we found in this notebook.** The properties that motivate the design hold exactly when checked numerically (spectral norm ≤ 1, closure under products, Birkhoff decomposition); unconstrained matrices that drift by ~15 % per layer reach a median composite gain around 12× in 60 layers while the doubly stochastic ones stay within 5 % of 1; 20 Sinkhorn-Knopp iterations are tight for well-conditioned logits and visibly loose for sharp ones; and the unfused traffic of an `n = 4` stream is ~11× a plain residual's by the paper's own table.

**What is still open.** The paper validates doubly stochastic constraints only, on MoE language models up to 27B parameters, and proposes other manifolds as future work. Its Sinkhorn-Knopp budget (`t_max = 20`) is not ablated; its overhead is reported as a single 6.7 % figure without a per-measure breakdown; and the benchmark table has no variance estimates. Our toy experiment is far too small to speak to any of these.

**Credits.** The method and all paper results are by Zhenda Xie, Yixuan Wei, Huanqi Cao and colleagues at DeepSeek-AI; the idea of Hyper-Connections is from Zhu et al.; the projection is the 1967 algorithm of Richard Sinkhorn and Paul Knopp, and the polytope is Garrett Birkhoff's. Errors in this notebook are ours.

### Exercises
1. Sweep `t_max ∈ {1, 2, 5, 10, 20, 50}` in Section 4 and plot composite backward gain after 60 layers against `t_max`. Where does it stop mattering for logit std 1 and 3?
2. Replace the sigmoid in `H_pre` by a softmax over the `n` streams (so the pre-mapping is a convex combination). Does the toy model of Section 7 train differently?
3. Initialize `b_res = κI` with `κ = 0` (uniform mixing) instead of 10. How much does the first training step change the output? What does that say about why initialization near the identity matters?
4. Implement a different manifold (for example, orthogonal matrices through a Cayley transform) and compare gain, mixing and cost with Sinkhorn-Knopp.

In [ ]:
print(f"{'#':>2s}  {'status':6s}  check")
for i, (n_, ok, d_) in enumerate(CHECKS, 1): print(f"{i:2d}  {'PASS' if ok else 'FAIL':6s}  {n_}")
failed = [n_ for n_, ok, _ in CHECKS if not ok]; print(f"\n{len(CHECKS) - len(failed)} passed, {len(failed)} failed")
assert not failed, failed